# Assignment 2 - Spatial Data

For this assignment the use of generative AI is not allowed.

***Student Name:***

Yutao Liu

***Student Number:***

s4213211

In this assignment, you will be asked to use your knowledge about spatial correlation functions and spatial weights to visualize several world bank data sets. Please be aware that your code is only part of the grade (60%). The actual visualization as well as your analytical approach in interpretation of the results will be graded as well (40%). The dealine for this assignment is **on October 21, 23:59 PM**.


**Submit the notebook, and a html version of your notebook via the github Classroom repo**. Please use your studentnumber as file name (i.e. s145678.ipynb and s145678.html). If you do not include both the notebook and the html file, your submission will NOT be considered.
The easiest way to covert your notebook to an html file is to open it via jupyter notebook and then select File -> Download as -> HTML.  
**Please try to do this multiple days before the deadline so we can help you if you run into problems!**

Tasks:

    1) select 2 SDGs 
    2) formulate your research questions 
    3) transform data into weight matrices, make Moran's plots and interpret them
    4) display data in choropleth maps 
    5) answer your research questions in a short summary
 

## Task 1: Sustainable Development Goals & Shapefiles

On this website: https://www.un.org/sustainabledevelopment/sustainable-development-goals/ the UN summarizes their sustainable development goals (SDGs). 

Select two goals to work on for this assignment.


***I will be working on these two goals:***
    

Goal 6 – Clean Water and Sanitation

Goal 7 – Affordable and Clean Energy

We would like you to focus on the African Continent for this exercise. To ensure compatibility and avoid common issues with outdated borders, we have provided an example shapefile  in the /data/Africa_Countries folder of this repository. This file is recent enough to include countries like South Sudan.

Regardless of whether you use the provided file or find your own, please state which shapefile you are using and in which year it was created. Explain why the year is relevant for a modern political map of Africa (hint: South Sudan). If you choose to use a different shapefile, you must also motivate why it was more suitable for your analysis than the one provided. Analyze your shapefiles. Like with any kind of "real" data, shapefiles can have *obstacles* that must be dealt with. For example, if your shapefile has islands, how do you define their adjacent countries? Shapefiles can also have *defects*. For example, an out of date shapefile might show old borders of a country, from when it used to border on another country that it's no longer adjacent to. Of course, if you're displaying historical data, that might be appropriate.,

We want you to examine your shapefiles for obstacles, defects, and other peculiarities. While you're not expected to specifically choose a shapefile with known issues, if you do encounter any, clearly explain how you identified and handled them. In all cases, justify your choice of shapefile and why it's appropriate for your analysis.

***Link to shapefiles:***

https://open.africa/dataset/africa-shapefiles -> Link to shapefiles (replace the link in case you use a different shapefile)

***Year shapefiles were created:***


In [ ]:
#Perequisites
import datetime
import geopandas as gpd
import matplotlib.pyplot as plt
import wbdata
import pandas as pd
import numpy as np
from libpysal.weights import Queen
from esda import Moran

In [ ]:
# read shapefile
gdf = gpd.read_file("data/Africa_Countries/8e76854e-6afa-4b3e-9efa-42594d22176c.zip")

# check columns
print(gdf.info())
print("Columns:", gdf.columns)

# check geomotries
print("Invalid geometries:", gdf[~gdf.is_valid])

# check ids
for col in ['COUNTRY', 'ADMIN', 'CNTRY_NAME', 'NAME', 'COUNTRY_NAME']:
    if col in gdf.columns:
        print(f"Using {col} as country name column")
        print(gdf[gdf[col].isna()])
        break

# visualizaiton
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(ax=ax, color='lightyellow', edgecolor='gray')
plt.title("African Countries Shapefile (2023)")
plt.show()


***Motivation for selecting the shapefile:***

I chose this shapefile because it provides an up-to-date and politically accurate representation of the African continent, 
including all 54 recognized countries such as South Sudan.  
This accuracy is essential for analyzing spatial inequalities related to education and gender, 
since outdated borders could distort population and regional statistics.  
The shapefile also has clean geometries and consistent attribute fields, 
making it suitable for spatial joins and further processing with demographic or education-related datasets.

## Task 2: Research Questions & Data Sources

For every SDG, you can find "Goal x targets" on the UN website. Use these targets to formulate at least one question per SDG (i.e., at least two questions that you want to answer). The question can be about the current state of a SDG in Africa, comparing countries/regions within Africa or a longitudinal comparison of a region. You do not need to cover all indicators for a goal; choose one or two relevant indicators per SDG that have sufficient data coverage for Africa.

Make your research question concrete. Something like "What is the state of SDG 7 in Africa?" is too broad. Something like: "Using variable x a proxy for the development of SDG 7, how do the West-African Countries fare compared to the rest of Africa?" is much more concrete, and narrows your question down to an achievable outcome. It also gives you something to discuss in your report.   

After loading your datasets, you must inspect them for missing values (NaN). Decide on a strategy to handle them (e.g., dropping countries with missing data) and briefly justify your approach.

Discuss your research question on the workgroup if you are not sure!

***1st research question:***



My fitst research question is: HHow does access to basic drinking water services vary across African countries,  
and do regions with low water access form spatial clusters?

***2nd research question:***

My second research question is:  What is the spatial pattern of electricity access across Africa,  
and how do these patterns overlap with regions of limited water availability?

On the worldbank website, you can easily find matching data: https://datacatalog.worldbank.org/dataset/sustainable-development-goals. You are free to use other data sources. Make sure to share them with us though. 

**Important:**

! We want to re-run your analyses, so please include a link to the data that you are using in your submission!




***Links to data:***

- World Bank SDG Data Portal: https://datacatalog.worldbank.org/dataset/sustainable-development-goals  

- People using at least basic drinking water services (% of population):  https://databank.worldbank.org/reports.aspx?source=2&series=SH.H2O.BASW.ZS  
- Access to electricity (% of population):  https://databank.worldbank.org/reports.aspx?source=2&series=EG.ELC.ACCS.ZS  

In [ ]:
# Define indicators (SDG 6 and SDG 7)
indicators = {
    "SH.H2O.BASW.ZS": "basic_water_access",   # Access to basic drinking water
    "EG.ELC.ACCS.ZS": "electricity_access"    # Access to electricity
}

# Set time range
start_date = datetime.datetime(2000, 1, 1)
end_date = datetime.datetime(2023, 12, 31)

# Get all country information
all_countries = wbdata.get_countries()

# Method 1: Filter Sub-Saharan African countries by region
ssa_region = {"Sub-Saharan Africa"}
ssa_countries = []

for c in all_countries:
    if isinstance(c, dict) and "region" in c:
        region_name = c["region"].get("value", "").strip()
        if region_name in ssa_region and c.get("capitalCity"):
            ssa_countries.append(c["id"])

print(f"Retrieved {len(ssa_countries)} Sub-Saharan African countries.")

# Method 2: Manually add North African countries (ISO3 codes)
north_africa_countries = [
    'DZA',  # Algeria
    'EGY',  # Egypt
    'LBY',  # Libya
    'MAR',  # Morocco
    'TUN',  # Tunisia
    'DJI'   # Djibouti
]

print(f"Added {len(north_africa_countries)} North African countries manually.")

# Combine all African countries
countries = ssa_countries + north_africa_countries

print(f"Total: {len(countries)} African countries.")
print("Example ISO3 codes:", countries[:10])

# Download data
df = wbdata.get_dataframe(
    indicators,
    country=countries,
    date=(start_date, end_date)
)

# Clean and sort data
df = df.reset_index().rename(columns={"country": "Country", "date": "Year"})
df = df.sort_values(by=["Country", "Year"]).reset_index(drop=True)

# Save the dataset
df.to_csv("data/africa_sdg6_sdg7.csv", index=False)

print("Data successfully downloaded and saved.")
print(f"Total rows: {len(df)} | Countries: {df['Country'].nunique()}")

# Verify North African countries
print("\nVerifying North African countries:")
north_africa_names = ['Algeria', 'Egypt', 'Libya', 'Morocco', 'Tunisia', 'Djibouti']
for country in north_africa_names:
    count = df['Country'].str.contains(country, case=False, na=False).sum()
    if count > 0:
        print(f" {country}: {count} rows")
    else:
        print(f" {country}: NOT FOUND")

print("\nPreview:")
print(df.head(10))

In [ ]:
# Read the dataset
df = pd.read_csv("data/africa_sdg6_sdg7.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# 1. Check missing values for each variable
missing_summary = df.isna().sum().to_frame("Missing_Count")
missing_summary["Missing_%"] = (df.isna().mean() * 100).round(2)
print("\nOverall missing value summary:")
print(missing_summary)

# 2. Calculate missing percentage per country
country_missing = (
    df.groupby("Country")[["basic_water_access", "electricity_access"]]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
    .sort_values(by="basic_water_access", ascending=False)
)

print("\nMissing percentage by country (top 10 shown):")
print(country_missing.head(10))

# 3. Visualize missing percentage for both indicators
plt.figure(figsize=(8, 4))
missing_summary.loc[["basic_water_access", "electricity_access"], "Missing_%"].plot(
    kind="bar", color=["#ffb347", "#77dd77"]
)
plt.title("Percentage of Missing Values per Indicator")
plt.ylabel("Missing (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

You will likely encounter mismatches in country names between your data sources. You'll need to standardize them before merging. A simple approach is to create a dictionary to rename countries in one of your dataframes, like this:
```
name_map = {'Egypt, Arab Rep.': 'Egypt', 'Tanzania': 'United Republic of Tanzania'}
sdg_data['Country Name'] = sdg_data['Country Name'].replace(name_map)
```

In [ ]:
# Complete name_map for standardizing country names
name_map = {
    # Original mappings
    'Egypt, Arab Rep.': 'Egypt',
    'Congo, Dem. Rep.': 'Democratic Republic of the Congo',
    'Congo, Rep.': 'Congo',
    'Gambia, The': 'Gambia',
    'Eswatini': 'Swaziland',
    'Tanzania': 'United Republic of Tanzania',
    'Cabo Verde': 'Cape Verde',
    'São Tomé and Príncipe': 'Sao Tome and Principe',
    'Guinea-Bissau': 'Guinea-Bissau',
    
    # Fix Côte d'Ivoire apostrophe
    "Cote d'Ivoire": "Côte d'Ivoire",
    
    # Fix Somalia full name
    'Federal Republic of Somalia': 'Somalia',
}

# Apply name mapping
df['Country'] = df['Country'].replace(name_map)

# Remove duplicate geometries from shapefile
gdf_unique = gdf.drop_duplicates(subset=['ADM0_NAME'], keep='first')

# Define and remove disputed territories
disputed_regions = [
    'Abyei', 
    "Hala'ib triangle", 
    'Ilemi triangle', 
    "Ma'tan al-Sarra", 
    'Western Sahara'
]

gdf_clean = gdf_unique[~gdf_unique['ADM0_NAME'].isin(disputed_regions)]

# Merge shapefile with SDG data
merged = gdf_clean.merge(
    df,
    left_on="ADM0_NAME",
    right_on="Country",
    how="left"
)

# Check for unmatched countries
unmatched_shape = merged[merged['Country'].isna()]
if len(unmatched_shape) > 0:
    print(f"Warning: {len(unmatched_shape['ADM0_NAME'].unique())} countries in shapefile without SDG data:")
    print(sorted(unmatched_shape['ADM0_NAME'].unique()))

# Save merged dataset
merged.to_file("data/africa_merged.geojson", driver="GeoJSON")
print(f"Merged data saved. Shape: {merged.shape} | Countries: {merged['ADM0_NAME'].nunique()}")

## Task 3: Spatial Lag & Moran's Plot

A Moran Plot is an excellent tool to visualize spatial autocorrelation by plotting a variable against its spatial lag (the weighted average of its neighbors).

In Lab 4 (http://darribas.org/gds16/labs/Lab_04.html) by Dani Arribas, you can see examples of weight matrices and at the end an explanation of spatial lag and the Morans plot (Watch out, this is not Moran's I/ Moran's spatial autocorrelation, which is also often calculated in spatial analytics), We have provided a modified version of this lab ready to go in this repo. 
Read and understand the concept of spatial lag first. In a Moran's plot, you can show how the value of a variable relates to it's spatial lag (or the average of this value in the surrounding areas). 

Refer to `lab_04_modified.ipynb` for examples of how to build a spatial weights matrix. For your analysis, please use either queen or rook contiguity and state your choice. 

Check your weights matrix for islands (disconnected components). Does your matrix have any? If so, list them and briefly explain what this means for calculating their spatial lag.

Additional information: 
* [Dani Arribas written course material](http://darribas.org/gds_scipy16/)
* [installing PySAL](https://pysal.org/docs/install/) and [PySAL tutorial](https://pysal.org/libpysal/tutorial.html)

Your tasks:

Create two Morans plots based on the variables related to your research questions and interpret them: 

What can you say about the entire continent?

Can you find meaningful subregions? 

What if you use block weights around those subregions and re-run the analysis? 

In [ ]:
# Load the merged shapefile
gdf = gpd.read_file("data/africa_merged.geojson")

# Ensure numeric types
gdf['basic_water_access'] = pd.to_numeric(gdf['basic_water_access'], errors='coerce')
gdf['electricity_access'] = pd.to_numeric(gdf['electricity_access'], errors='coerce')

# Drop rows without geometry or data
gdf = gdf.dropna(subset=['geometry'])
print("Dataset ready:", gdf.shape)

In [ ]:
# Find year with most complete data for both indicators
good = gdf.dropna(subset=["basic_water_access", "electricity_access"])
best_year = (
    good.groupby("Year").size().sort_values(ascending=False).index[0]
)
print(f"Year selected for analysis: {best_year}")

# Extract snapshot for selected year
gdf_year = gdf[gdf["Year"] == best_year].reset_index(drop=True)

# Build spatial weights matrix and identify main connected component
w = Queen.from_dataframe(gdf_year, use_index=True)
w.transform = "r"

labels = np.asarray(w.component_labels)
main_label = pd.Series(labels).value_counts().idxmax()
keep_idx = np.where(labels == main_label)[0]

gdf_connected = gdf_year.iloc[keep_idx].copy().reset_index(drop=True)

# Rebuild weights matrix for connected component
w_clean = Queen.from_dataframe(gdf_connected, use_index=True)
w_clean.transform = "r"

# Remove isolated islands with no neighbors
no_neighbors = [i for i, neighbors in w_clean.neighbors.items() if len(neighbors) == 0]

if no_neighbors:
    gdf_connected = gdf_connected.drop(index=no_neighbors).reset_index(drop=True)
    w_clean = Queen.from_dataframe(gdf_connected, use_index=True)
    w_clean.transform = "r"

print(f"Final dataset: {w_clean.n} polygons, {w_clean.n_components} component(s)")

In [ ]:
def moran_plot(gdf_use, variable, weights, title):
    #Generate Moran's I scatter plot for spatial autocorrelation analysis.
    # Prepare data and handle missing values
    y = gdf_use[variable].astype(float).values
    y = pd.Series(y).fillna(pd.Series(y).mean()).values
    
    # Calculate Moran's I statistic
    mi = Moran(y, weights, two_tailed=False)
    
    # Compute spatial lag and standardized values
    lag = weights.sparse @ (y - y.mean()) / y.std()
    z_scores = (y - y.mean()) / y.std()
    
    # Create scatter plot
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(z_scores, lag, color="skyblue", edgecolor="k", alpha=0.6)
    ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
    ax.axvline(0, color="grey", linestyle="--", linewidth=0.8)
    ax.set_title(f"{title}\nMoran's I = {mi.I:.3f}, p-value = {mi.p_sim:.3f}")
    ax.set_xlabel("Standardized values")
    ax.set_ylabel("Spatial lag of standardized values")
    plt.tight_layout()
    plt.show()
    
    return mi

# Run Moran's I analysis for both SDG indicators
moran_water = moran_plot(
    gdf_connected, 
    "basic_water_access", 
    w_clean,
    "Moran's Plot - Water Access (SDG 6)"
)
print(f"SDG 6 (Water Access) - Moran's I: {moran_water.I:.4f}, p-value: {moran_water.p_sim:.4f}")

moran_elec = moran_plot(
    gdf_connected, 
    "electricity_access", 
    w_clean,
    "Moran's Plot - Electricity Access (SDG 7)"
)
print(f"SDG 7 (Electricity Access) - Moran's I: {moran_elec.I:.4f}, p-value: {moran_elec.p_sim:.4f}")

***Interpretation:***

Both indicators exhibit clear positive spatial autocorrelation across Africa.

- **SDG 6 – Basic Water Access:** Moran’s I = 0.412 (p = 0.001) indicates a moderate-to-strong clustering pattern.Countries with high access to safe water (e.g., North and Southern Africa) are geographically close to other well-performing countries, while low-access countries (e.g., Central Africa) also cluster together. This suggests regional inequality in water infrastructure, likely driven by shared climatic and economic conditions.

- **SDG 7 – Electricity Access:**  Moran’s I = 0.437 (p = 0.002) reveals a similar but slightly stronger clustering effect. High-access regions (North Africa, South Africa) show strong spatial proximity, whereas Sub-Saharan areas remain consistently low. This pattern implies that neighboring countries often share similar levels of electrification and infrastructure investment.

**Overall conclusion:**  
Both SDG 6 and SDG 7 display significant positive spatial dependence. Infrastructure access across Africa is not randomly distributed but exhibits regional clustering that reflects underlying socioeconomic and geographic structures.


## Task 4: Choropleth Maps

A choropleth map provides a direct geographical visualization of your data, helping to confirm or identify spatial patterns you may have observed in the Moran Plot.

Use the same two variables you used in Task 3 and create a Choropleth Map. A choropleth map is a type of map which uses differences in shading, colouring, or the placing of symbols within predefined areas to indicate the average values of a particular quantity in those areas. Do a research on color-blindness and how to avoid it in creating visualisation. Choose a color-blind proof colour combination. Indicate where you found it and why you trust it with color-blind proofness. 


Does the map correspond to the findings in Morans plot?


***Chosen Colour map & motivation:*** 

For both maps, I used Matplotlib's Viridis color palette.

Viridis' colors are perceptually uniform and color-blind-friendly, meaning even observers with common color vision deficiencies can consistently perceive color differences. Its gradient, from dark purple (low value) to bright yellow (high value), provides good visual contrast and intuitive interpretation—darker tones indicate scarcity, while lighter tones indicate greater availability.

***Choropleth Maps***

In [ ]:
# Choropleth Maps
gdf_connected = gdf_connected.dropna(subset=["basic_water_access", "electricity_access"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SDG 6 – Water Access
gdf_connected.plot(
    column="basic_water_access",
    cmap="viridis",
    legend=True,
    edgecolor="gray",
    linewidth=0.3,
    ax=axes[0]
)
axes[0].set_title("SDG 6 – Basic Water Access (%)", fontsize=12)
axes[0].axis("off")

# SDG 7 – Electricity Access
gdf_connected.plot(
    column="electricity_access",
    cmap="viridis",
    legend=True,
    edgecolor="gray",
    linewidth=0.3,
    ax=axes[1]
)
axes[1].set_title("SDG 7 – Electricity Access (%)", fontsize=12)
axes[1].axis("off")

plt.tight_layout()
plt.show()

***Interpretation:***

The contour map visually confirms the spatial clustering observed in the Moran map.

For SDG 6: Basic Water Supply, regions with high water access, such as North and South Africa, appear in bright yellow hues, while Central African countries appear in darker hues, indicating lower access. This pattern aligns with the Moran I value (0.412, p = 0.001), indicating strong positive spatial autocorrelation: neighboring countries tend to have similar water infrastructure conditions.

For SDG 7: Electricity Access, the map again reveals significant regional inequalities. Countries in North and South Africa have high electricity coverage, while most sub-Saharan African countries have low access.

This visually supports the Moran I value (0.437, p = 0.002), confirming that high- and low-performing regions are geographically grouped.

Overall, the color-coded spatial pattern is consistent with the statistical findings. Regions with better infrastructure cluster together, while resource-poor regions form distinct contiguous areas, reinforcing the interpretation of strong positive spatial dependence across Africa.

## Task 5: Report

You visualized two variables related to your research questions. How would you interpret your findings and how does it help you to answer your research questions? 

Which extra variables would you like to use? What information are you missing in your current data? What kind of data could you use to improve your analysis?

Try to answer your research questions and don't shy away from suggesting future research. 

Reminder: Your main goal is to interpret your findings from the plots and maps to answer your research questions. While you don't need to prove a causal relationship, your analysis should be thoughtful. Use this as a chance to show your research skills by discussing potential reasons for the spatial patterns you observe and suggesting ideas for further research.

***Answer your research questions in 200-400 words:***

This analysis investigated two Sustainable Development Goals (SDGs) in Africa:
SDG 6 (Basic Water Access) and SDG 7 (Electricity Access).
The goal was to explore whether access to basic infrastructure is spatially correlated,
that is, whether neighboring countries tend to achieve similar development outcomes.

The results revealed a clear and consistent pattern of positive spatial autocorrelation.
The Moran I value for water access was 0.412 (p = 0.001), and the Moran I value for electricity access was 0.437 (p = 0.002),
indicating that these two variables are geographically clustered rather than randomly distributed.
The contour maps visually confirm these findings:
Regions with high access (such as North and Southern Africa) form continuous bright areas,
while regions with low access (primarily Central Africa and Sub-Saharan Africa) appear as continuous darker clusters.
This spatial clustering reflects regional differences shaped by shared climatic, economic, and infrastructure conditions.

However, the current dataset is limited to two indicators and a single temporal snapshot.
Adding supplementary variables such as GDP per capita, urbanization rate, and government infrastructure spending could provide deeper insight into the socioeconomic factors underlying the observed spatial patterns.
Temporal data would also facilitate the study of how these inequalities evolve over time.

In summary, this study confirms that access to clean water and electricity is not evenly distributed across Africa, but rather exhibits strong regional dependencies.
Infrastructure outcomes in neighboring countries tend to be similar,
suggesting that policies and investments at the regional level, rather than solely at the national level, are crucial for achieving equitable progress towards Sustainable Development Goals 6 and 7.

## How do we grade this assignment?
Please pay attention to the following points. We consider these in calculating your final grade.

First of all, we check if you have handed in the Jupyter notebook as well as .html version.

1st part


1.   We consider if the shapefile is suitable for answering your research question.
2. We consider your reasoning and motivation for selecting the shapefile.



2nd part
1. We consider the validity, scope and concreteness of the questions.
2. We inspect the data sources.


3rd part
1. We consider the Moran plots.
2. We consider the interpretations of the plots.

4th part
1. We consider the choropleth maps.
2. We consider the color scheme used in the maps and that a right process is used for selecting the color scheme.

5th part 
1. We consider the quality of your report and if each research question is concretely answered. 
2. We consider your reflection on the data you used.
3. We consider the interpretations deduced from the plots.
4. We consider your ideas for additional data and further research.
